# Explore datalake HTTP API

Call the **datalake** service (canonical historical data, Tiingo-backed ingestion).

**Prerequisites**
- Run datalake locally (e.g. Docker Compose maps port **8000**) or point `DATALAKE_BASE_URL` at your instance.
- Use this service’s virtualenv: from repo root, `make venv-service SERVICE=datalake`, activate, then `pip install jupyterlab ipykernel httpx` if needed.
- See root `README.md` for Jupyter setup.

**Endpoints used below:** `GET /health`, `GET /tickers`, `GET /bars/{symbol}` (optional query: `start_date`, `end_date`, `limit`).

Add more `.ipynb` files in this directory for ad-hoc experiments; keep `explore.ipynb` as the shared template.

In [ ]:
import json
import os

import httpx

BASE_URL = os.environ.get("DATALAKE_BASE_URL", "http://localhost:8000").rstrip("/")
print("BASE_URL =", BASE_URL)

In [ ]:
with httpx.Client(timeout=30.0) as client:
    r = client.get(f"{BASE_URL}/health")
    r.raise_for_status()
    print(json.dumps(r.json(), indent=2))

In [ ]:
with httpx.Client(timeout=30.0) as client:
    r = client.get(f"{BASE_URL}/tickers")
    r.raise_for_status()
    tickers = r.json()
    print(f"tickers count: {len(tickers)}")
    print(json.dumps(tickers[:5], indent=2) if tickers else "(empty)")

In [ ]:
symbol = "AAPL"
params = {"limit": 5}

with httpx.Client(timeout=30.0) as client:
    r = client.get(f"{BASE_URL}/bars/{symbol}", params=params)
    if r.status_code == 404:
        print("404: no bars (ticker missing or no data)")
    else:
        r.raise_for_status()
        bars = r.json()
        print(json.dumps(bars, indent=2, default=str))

## Next steps

- Try `GET /bars/{symbol}` with `start_date` and `end_date` query params (ISO dates).
- When auth exists, attach headers on the `httpx.Client`.
- Duplicate cells to probe `POST /tickers` or `POST /bars/backfill` with care in non-prod only.